# Shared Overlaps And Conversion Pools

This notebook distinguishes three related objects:

- raw cross-modal overlaps in `data/manifests/intersections/`
- trainable shared conversion pools in `data/manifests/conversion_pools/`
- fixed shared conversion split manifests in `data/manifests/splits/conversion/`


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))


            from pathlib import Path
            import pandas as pd

            from src.data.image_manifest import conversion_split_dir_from_config
            from src.eval_utils import load_config

            config = load_config(ROOT / "config.yaml")


In [ ]:
def load_ids(path):
    return [line.strip() for line in Path(path).read_text().splitlines() if line.strip()]

def concept_count(image_ids):
    return len({image_id.rsplit("_", 1)[0] for image_id in image_ids})

raw_rows = []
conversion_rows = []
for name in ["eeg_meg", "eeg_fmri", "fmri_meg", "eeg_fmri_meg"]:
    raw_path = ROOT / "data" / "manifests" / "intersections" / f"{name}.txt"
    pool_path = ROOT / "data" / "manifests" / "conversion_pools" / f"{name}.txt"
    split_dir = conversion_split_dir_from_config(config, shared_manifest_path=pool_path)

    raw_ids = load_ids(raw_path)
    pool_ids = load_ids(pool_path)
    train_ids = load_ids(split_dir / "train.txt")
    val_ids = load_ids(split_dir / "val.txt")
    test_ids = load_ids(split_dir / "test.txt")
    excluded_ids = load_ids(split_dir / "excluded.txt")

    raw_rows.append(
        {
            "pool": name,
            "raw_overlap_images": len(raw_ids),
            "raw_overlap_concepts": concept_count(raw_ids),
        }
    )
    conversion_rows.append(
        {
            "pool": name,
            "conversion_pool_images": len(pool_ids),
            "conversion_pool_concepts": concept_count(pool_ids),
            "train_images": len(train_ids),
            "train_concepts": concept_count(train_ids),
            "val_images": len(val_ids),
            "val_concepts": concept_count(val_ids),
            "test_images": len(test_ids),
            "test_concepts": concept_count(test_ids),
            "excluded_images": len(excluded_ids),
        }
    )

raw_df = pd.DataFrame(raw_rows)
conversion_df = pd.DataFrame(conversion_rows)


In [ ]:
raw_df


In [ ]:
conversion_df
